    # Modelling DisasterWatch — baseline

    Baseline **klasifikasi jenis bencana (`kejadian`)**.

    Asumsi ini dibuat karena target bisnis belum ditentukan. Untuk use case prediksi korban,
    dampak, atau forecasting jumlah kejadian, ganti `TARGET` dan strategi split di bawah.
    

In [ ]:
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    from sklearn.model_selection import train_test_split
    
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    DATA = ROOT / "data" / "processed" / "bnpb_disaster_features.csv"
    df = pd.read_csv(DATA, encoding="utf-8-sig", parse_dates=["date"])
    TARGET = "kejadian"
    df = df.dropna(subset=[TARGET]).sort_values("date")
    print(df.shape, df[TARGET].value_counts())
    

## 1. Siapkan fitur dan split temporal

In [ ]:
    # Hindari kolom target dan kolom yang berpotensi membocorkan target.
    # Karena target adalah jenis bencana, fitur dampak tetap dikeluarkan dari baseline.
    feature_cols = ["year", "month", "day_of_year", "week_of_year", "day_of_week", "is_weekend",
                    "quarter", "provinsi", "kabkot", "longitude", "latitude", "has_coordinates",
                    "province_event_count", "kabkot_event_count"]
    feature_cols = [c for c in feature_cols if c in df.columns]
    X, y = df[feature_cols], df[TARGET]
    split = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    print("Train:", X_train.shape, "Test:", X_test.shape)
    

## 2. Pipeline preprocessing + model

In [ ]:
    categorical = [c for c in ["provinsi", "kabkot"] if c in X.columns]
    numerical = [c for c in feature_cols if c not in categorical]
    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numerical),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")),
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(classification_report(y_test, pred, zero_division=0))
    

In [ ]:
    labels = sorted(y_test.unique())
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=labels, xticks_rotation=45, cmap="Blues")
    plt.tight_layout()
    

## 3. Validasi dan langkah lanjutan

- Bandingkan dengan baseline `DummyClassifier`.
- Gunakan rolling/expanding time-series validation.
- Tentukan target final: klasifikasi jenis bencana, regresi korban, atau forecasting per wilayah.
- Audit leakage: jangan memakai fitur yang baru diketahui setelah kejadian jika model dimaksudkan untuk peringatan dini.